# 데코레이터 패턴 구현

## 1. 추상 구성 요소 (Component)

In [4]:
abstract class Beverage {
    enum class Size { TALL, GRANDE, VENTI }
    
    open var size: Size = Size.TALL
    open var description: String = "Unknown Beverage"
    
    abstract fun cost(): Double
}

## 2. 구상 구성 요소 (Concrete Component) - 음료

In [5]:
class Espresso : Beverage() {
    init { description = "에스프레소" }
    override fun cost() = 1.99
}

class DarkRoast : Beverage() {
    init { description = "다크 로스트" }
    override fun cost() = 0.99
}

class HouseBlend : Beverage() {
    init { description = "하우스 블렌드" }
    override fun cost() = 0.89
}

class Decaf : Beverage() {
    init { description = "디카페인" }
    override fun cost() = 1.05
}

## 3. 추상 데코레이터 (Decorator)

In [6]:
abstract class CondimentDecorator(protected val beverage: Beverage) : Beverage() {
    override var size: Size
        get() = beverage.size
        set(value) { beverage.size = value }
}

## 4. 구상 데코레이터 (Concrete Decorator) - 첨가물

In [7]:
class Mocha(beverage: Beverage) : CondimentDecorator(beverage) {
    override var description: String
        get() = "${beverage.description}, 모카"
        set(value) {}
    
    override fun cost() = beverage.cost() + 0.20
}

class Whip(beverage: Beverage) : CondimentDecorator(beverage) {
    override var description: String
        get() = "${beverage.description}, 휘핑크림"
        set(value) {}
    
    override fun cost() = beverage.cost() + 0.10
}

class Milk(beverage: Beverage) : CondimentDecorator(beverage) {
    override var description: String
        get() = "${beverage.description}, 우유"
        set(value) {}
    
    override fun cost() = beverage.cost() + 0.10
}

// 사이즈별 가격 차등 적용
class Soy(beverage: Beverage) : CondimentDecorator(beverage) {
    override var description: String
        get() = "${beverage.description}, 두유"
        set(value) {}
    
    override fun cost(): Double {
        val extra = when (beverage.size) {
            Beverage.Size.TALL -> 0.10
            Beverage.Size.GRANDE -> 0.15
            Beverage.Size.VENTI -> 0.20
        }
        return beverage.cost() + extra
    }
}

## 5. 실행 예제

In [8]:
println("=== 스타버즈 커피 주문 ===")
println()

// 1. 에스프레소 주문
val beverage1 = Espresso()
println("${beverage1.description} \$${beverage1.cost()}")

// 2. 다크 로스트 + 더블 모카 + 휘핑크림
var beverage2: Beverage = DarkRoast()
beverage2 = Mocha(beverage2)
beverage2 = Mocha(beverage2)  // 더블 모카!
beverage2 = Whip(beverage2)
println("${beverage2.description} \$${beverage2.cost()}")

// 3. 하우스 블렌드 + 두유 + 모카 + 휘핑크림
var beverage3: Beverage = HouseBlend()
beverage3 = Soy(beverage3)
beverage3 = Mocha(beverage3)
beverage3 = Whip(beverage3)
println("${beverage3.description} \$${beverage3.cost()}")

=== 스타버즈 커피 주문 ===

에스프레소 $1.99
다크 로스트, 모카, 모카, 휘핑크림 $1.49
하우스 블렌드, 두유, 모카, 휘핑크림 $1.29


## 6. 사이즈별 가격 차등 테스트

In [9]:
println("=== 사이즈별 두유 가격 ===")
println()

for (size in Beverage.Size.values()) {
    var beverage: Beverage = DarkRoast()
    beverage.size = size
    beverage = Soy(beverage)
    
    println("$size: ${beverage.description} \$${"%.2f".format(beverage.cost())}")
}

=== 사이즈별 두유 가격 ===

TALL: 다크 로스트, 두유 $1.09
GRANDE: 다크 로스트, 두유 $1.14
VENTI: 다크 로스트, 두유 $1.19


## 7. Java I/O 데코레이터 예제

In [10]:
import java.io.*

class LowerCaseInputStream(input: InputStream) : FilterInputStream(input) {
    override fun read(): Int {
        val c = super.read()
        return if (c == -1) c else Character.toLowerCase(c)
    }
    
    override fun read(b: ByteArray, off: Int, len: Int): Int {
        val result = super.read(b, off, len)
        for (i in off until off + result) {
            b[i] = Character.toLowerCase(b[i].toInt().toChar()).code.toByte()
        }
        return result
    }
}

In [11]:
val testText = "I know the Decorator Pattern therefore I RULE!"

val inputStream = LowerCaseInputStream(
    BufferedInputStream(
        ByteArrayInputStream(testText.toByteArray())
    )
)

println("원본: $testText")
print("변환: ")

inputStream.use { stream ->
    var c: Int
    while (stream.read().also { c = it } >= 0) {
        print(c.toChar())
    }
}
println()

원본: I know the Decorator Pattern therefore I RULE!
변환: i know the decorator pattern therefore i rule!


## 8. Java Collections 데코레이터 예제


### Collections.checkedList - 타입 체크 데코레이터

런타임에 타입 안전성을 보장하는 데코레이터

In [13]:
import java.util.*

// 일반 ArrayList - 제네릭은 컴파일 타임에만 체크
val rawList = ArrayList<Any>()
rawList.add("String")
rawList.add(123)  // 아무거나 들어감
println("rawList: $rawList")

// checkedList로 감싸면 런타임에도 타입 체크
val checkedList = Collections.checkedList(ArrayList<String>(), String::class.java)
checkedList.add("Hello")
checkedList.add("World")
println("checkedList: $checkedList")

// 잘못된 타입 추가 시도 - 런타임 에러 발생
try {
    (checkedList as MutableList<Any>).add(123)
} catch (e: ClassCastException) {
    println("타입 체크 에러: ${e.message}")
}

rawList: [String, 123]
checkedList: [Hello, World]
타입 체크 에러: Attempt to insert class java.lang.Integer element into collection with element type class java.lang.String


### Collections.unmodifiableList - 불변 데코레이터

수정 불가능하게 만드는 데코레이터

In [ ]:
val mutableList = mutableListOf("A", "B", "C")
println("원본 리스트: $mutableList")

// unmodifiableList로 감싸면 읽기 전용
val readOnlyList = Collections.unmodifiableList(mutableList)
println("읽기 전용 리스트: $readOnlyList")

// 수정 시도 - 에러 발생
try {
    readOnlyList.add("D")
} catch (e: UnsupportedOperationException) {
    println("수정 불가 에러: ${e.javaClass.simpleName}")
}

// 원본은 여전히 수정 가능 (주의!)
mutableList.add("D")
println("원본 수정 후: $mutableList")
println("읽기 전용도 반영됨: $readOnlyList")

## 9. Spring Filter / Interceptor

서블릿 필터는 데코레이터 패턴과 유사하게, 여러 개를 체인으로 연결해서 요청/응답에 기능을 추가함.

```kotlin
// 서블릿 필터 체인 - 데코레이터처럼 여러 개 중첩
class LoggingFilter : Filter {
    override fun doFilter(request, response, chain: FilterChain) {
        println("요청 시작: ${request.requestURI}")
        chain.doFilter(request, response)  // 다음 필터로 위임
        println("요청 종료: ${request.requestURI}")
    }
}

class AuthFilter : Filter {
    override fun doFilter(request, response, chain: FilterChain) {
        if (isAuthenticated(request)) {
            chain.doFilter(request, response)  // 다음 필터로 위임
        } else {
            response.sendError(401, "Unauthorized")
        }
    }
}
```

```
요청 흐름: Client → LoggingFilter → AuthFilter → Controller
                         ↓              ↓            ↓
                      (로깅)        (인증체크)     (비즈니스 로직)
```

- 각 필터는 `chain.doFilter()`로 다음 필터에게 **위임**
- 데코레이터처럼 **여러 개 중첩** 가능
- 기존 코드 수정 없이 **기능 추가** 가능